In [ ]:
# --------------------------------------------------
# 04 - Fraud Detection Model Training
# --------------------------------------------------

import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# --------------------------------------------------
# Load Config
# --------------------------------------------------

with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Config loaded")


# --------------------------------------------------
# Load Engineered Dataset
# --------------------------------------------------

df = pd.read_csv("../data/processed/creditcard_engineered.csv")

print("Dataset Shape:", df.shape)


# --------------------------------------------------
# Feature / Target Split
# --------------------------------------------------

TARGET = config["dataset"]["target_column"]

X = df.drop(TARGET, axis=1)
y = df[TARGET]

print("Features:", X.shape)
print("Target:", y.shape)


# --------------------------------------------------
# Train Test Split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=config["random_state"],
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


# --------------------------------------------------
# Logistic Regression
# --------------------------------------------------

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

log_acc = accuracy_score(y_test, y_pred_log)

print("\nLogistic Regression Results")
print("Accuracy:", log_acc)
print(classification_report(y_test, y_pred_log))


# --------------------------------------------------
# Random Forest
# --------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=50,
    random_state=config["random_state"]
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)

print("\nRandom Forest Results")
print("Accuracy:", rf_acc)
print(classification_report(y_test, y_pred_rf))


# --------------------------------------------------
# XGBoost
# --------------------------------------------------

xgb_model = XGBClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=6,
    random_state=config["random_state"],
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, y_pred_xgb)

print("\nXGBoost Results")
print("Accuracy:", xgb_acc)
print(classification_report(y_test, y_pred_xgb))


# --------------------------------------------------
# Confusion Matrix (Random Forest)
# --------------------------------------------------

cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


# --------------------------------------------------
# Model Comparison Table
# --------------------------------------------------

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [log_acc, rf_acc, xgb_acc]
})

print("\nModel Comparison")
print(results)

Config loaded
Dataset Shape: (454902, 31)
Features: (454902, 30)
Target: (454902,)
Train Shape: (363921, 30)
Test Shape: (90981, 30)

Logistic Regression Results
Accuracy: 0.9508798540354579
              precision    recall  f1-score   support

           0       0.93      0.97      0.95     45491
           1       0.97      0.93      0.95     45490

    accuracy                           0.95     90981
   macro avg       0.95      0.95      0.95     90981
weighted avg       0.95      0.95      0.95     90981

